In [7]:
import pandas as pd
import torch
from tqdm import tqdm
from transformers import BertTokenizer, BertForSequenceClassification, pipeline

# --- Step 1: Load tokenizer and model only once ---
print("🔄 Loading FinBERT model and tokenizer...")
model_path = "/Users/shokoufehnaseri/Library/CloudStorage/OneDrive-Personal/Master-Thesis/Master_Thesis/src/models/finbert_final_finetuned"
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Create sentiment analysis pipeline
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# --- Step 2: Loop over 20 CSV chunks ---
for i in range(1, 21):
    print(f"\n📥 Loading chunk {i}...")
    input_path = f"/Users/shokoufehnaseri/Library/CloudStorage/OneDrive-Personal/Master-Thesis/Master_Thesis/chunks/chunks/chunk_{i}.csv"
    output_path = f"/Users/shokoufehnaseri/Library/CloudStorage/OneDrive-Personal/Master-Thesis/Master_Thesis/chunks/sentiment_output_chunk_{i}.csv"

    try:
        df = pd.read_csv(input_path)

        # Run sentiment analysis
        print("🔍 Running sentiment analysis...")
        sentiments = []
        for text in tqdm(df["Text_Cleaned"].astype(str), desc=f"Processing chunk {i}"):
            try:
                sentiment = sentiment_pipeline(text)[0]['label']
            except Exception:
                sentiment = "ERROR"
            sentiments.append(sentiment)

        df["sentiment"] = sentiments

        # Save result
        df.to_csv(output_path, index=False)
        print(f"✅ Sentiment predictions saved to: {output_path}")

    except FileNotFoundError:
        print(f"⚠️ Chunk {i} not found. Skipping.")


Device set to use cpu


🔄 Loading FinBERT model and tokenizer...

📥 Loading chunk 1...
🔍 Running sentiment analysis...


Processing chunk 1: 100%|██████████| 183734/183734 [9:10:50<00:00,  5.56it/s]      


✅ Sentiment predictions saved to: /Users/shokoufehnaseri/Library/CloudStorage/OneDrive-Personal/Master-Thesis/Master_Thesis/chunks/sentiment_output_chunk_1.csv

📥 Loading chunk 2...
🔍 Running sentiment analysis...


Processing chunk 2: 100%|██████████| 183734/183734 [2:46:47<00:00, 18.36it/s]    


✅ Sentiment predictions saved to: /Users/shokoufehnaseri/Library/CloudStorage/OneDrive-Personal/Master-Thesis/Master_Thesis/chunks/sentiment_output_chunk_2.csv

📥 Loading chunk 3...
🔍 Running sentiment analysis...


Processing chunk 3:   0%|          | 112/183734 [00:03<1:22:53, 36.92it/s]


KeyboardInterrupt: 

In [10]:
import pandas as pd
import torch
from tqdm import tqdm
from transformers import BertTokenizer, BertForSequenceClassification
from datasets import Dataset
from torch.utils.data import DataLoader

# --- Step 1: Load model and tokenizer once ---
print("🔄 Loading FinBERT model and tokenizer...")
model_path = "/Users/shokoufehnaseri/Library/CloudStorage/OneDrive-Personal/Master-Thesis/Master_Thesis/src/models/finbert_final_finetuned"
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# --- Step 2: Process all 20 chunks ---
for i in range(9, 21):
    print(f"\n📥 Loading chunk {i}...")
    input_path = f"/Users/shokoufehnaseri/Library/CloudStorage/OneDrive-Personal/Master-Thesis/Master_Thesis/chunks/chunks/chunk_{i}.csv"
    output_path = f"/Users/shokoufehnaseri/Library/CloudStorage/OneDrive-Personal/Master-Thesis/Master_Thesis/chunks/sentiment_output_chunk_{i}.csv"

    try:
        df = pd.read_csv(input_path)
        df = df.dropna(subset=['Text_Cleaned'])  # remove missing text entries

        print("🔍 Tokenizing...")
        def preprocess(batch):
            return tokenizer(batch['Text_Cleaned'], padding='max_length', truncation=True, max_length=128)

        tweets_ds = Dataset.from_pandas(df[['Text_Cleaned']])
        tweets_ds = tweets_ds.map(preprocess, batched=True)
        tweets_ds.set_format(type='torch', columns=['input_ids', 'attention_mask'])

        loader = DataLoader(tweets_ds, batch_size=64)
        all_preds = []

        print("🔮 Running predictions...")
        for batch in tqdm(loader, desc=f"Predicting chunk {i}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            with torch.no_grad():
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                preds = torch.argmax(logits, axis=1)
                all_preds.extend(preds.cpu().numpy())

        label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
        df = df.iloc[:len(all_preds)].copy()  # match row count
        df['sentiment'] = [label_map[p] for p in all_preds]

        df.to_csv(output_path, index=False)
        print(f"✅ Saved sentiment_output_chunk_{i}.csv")

    except FileNotFoundError:
        print(f"⚠️ Chunk {i} not found. Skipping.")


🔄 Loading FinBERT model and tokenizer...

📥 Loading chunk 9...
🔍 Tokenizing...


Map:   0%|          | 0/183734 [00:00<?, ? examples/s]

🔮 Running predictions...


Predicting chunk 9: 100%|██████████| 2871/2871 [7:53:48<00:00,  9.90s/it]   


✅ Saved sentiment_output_chunk_9.csv

📥 Loading chunk 10...
🔍 Tokenizing...


...

✅ Saved sentiment_output_chunk_19.csv

📥 Loading chunk 20...
🔍 Tokenizing...


Map:   0%|          | 0/183733 [00:00<?, ? examples/s]

🔮 Running predictions...


Predicting chunk 20: 100%|██████████| 2871/2871 [8:04:11<00:00, 10.12s/it]     


✅ Saved sentiment_output_chunk_20.csv


In [12]:
import glob
import pandas as pd

all_outputs = glob.glob("/Users/shokoufehnaseri/Library/CloudStorage/OneDrive-Personal/Master-Thesis/Master_Thesis/chunks/sentiment_output_chunk_*.csv")
print("Found files:", len(all_outputs))  # Check how many files it found

if not all_outputs:
    raise FileNotFoundError("No sentiment output files found. Please check the path and filenames.")

df_merged = pd.concat((pd.read_csv(f) for f in all_outputs), ignore_index=True)
df_merged.to_csv("merged_sentiment_output.csv", index=False)
print("✅ Merged all sentiment outputs into 'merged_sentiment_output.csv'")


Found files: 20
✅ Merged all sentiment outputs into 'merged_sentiment_output.csv'


In [13]:
df= pd.read_csv('/Users/shokoufehnaseri/Library/CloudStorage/OneDrive-Personal/Master-Thesis/Master_Thesis/src/models/merged_sentiment_output.csv')

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3674675 entries, 0 to 3674674
Data columns (total 9 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   tweet_id      int64 
 1   writer        object
 2   created_at    object
 3   text          object
 4   comment_num   int64 
 5   retweet_num   int64 
 6   like_num      int64 
 7   Text_Cleaned  object
 8   sentiment     object
dtypes: int64(4), object(5)
memory usage: 252.3+ MB


In [15]:
df.isnull().sum()

tweet_id            0
writer          46986
created_at          0
text                0
comment_num         0
retweet_num         0
like_num            0
Text_Cleaned        0
sentiment           0
dtype: int64